# Trees: Conceptual Overview
A __rooted tree__ $T = \left\{\mathcal{V},\mathcal{E}\right\}$ composed of nodes $\mathcal{V}$ and edges $\mathcal{E}$, is a connected, acyclic structure with a  **root** node and a parent–child hierarchy. Each node has at most one parent and zero or more children, the only node without a parent being the **root**. The edges connect nodes, making trees ideal for modeling hierarchical data (e.g., file systems, org charts, etc). 

Let's look at some key elements of a tree structure:
> **Levels and Height**: Each node in the tree is assigned a level based on its distance from the root (node with no parent). The root is at level $h=0$, its children are at $h=1$, and so on. For any node, the number of edges on the path from the root to that node defines its level. The maximum level $\max_{v\in\mathcal{V}}\left(\texttt{level}(v)\right)$ of a tree is called the height $h$, here `h = 3`.

> **Leaves and Branches**: Leaves are nodes with no children; they can occur at any level of a tree. A branch is any path from the root down through its descendants to a leaf. One such root-to-leaf path is highlighted to illustrate how height corresponds to the length of the longest branch in the tree.

Before we get into the representation of trees, let's look at some applications.

<div>
    <center>
      <img
        src="figs/Fig-General-Tree-Example.svg"
        alt="General Tree Example"
        height="400"
        width="800"
      />
    </center>
  </div>

## Applications of Trees

Trees are widely used for various applications. In everyday life, trees can model various hierarchical relationships, such as: Family trees, organizational structures, file systems, and decision-making processes. In computational contexts, trees are used in data structures and algorithms, such as:

* __Binary Trees__: A tree where each node has at most two children, referred to as the left and right child. This structure is widely used, for example, in finance to represent future prices of an asset, where each node represents a possible price at a given time, and the left child represents a lower price while the right child represents a higher price. We can expand this concept to create more complex structures, e.g., where each node has $n$ children, leading to a __n-ary tree__.
* __Binary Search Tree (BST)__: A binary tree where the left child is less than the parent node, and the right child is greater than the parent node. This property allows for efficient searching, insertion, and deletion operations.
* __Heaps__: A specialized tree structure that satisfies the heap property, where the parent node is either greater than or equal to (max heap) or less than or equal to (min heap) its children. Heaps are commonly used in priority queues and sorting algorithms.
* __Tries__: A tree-like structure used for storing strings, where each node represents a character in the string. Tries are particularly useful for tasks such as autocomplete and spell checking, as they enable efficient prefix searching and retrieval of strings.

They provide a clear and intuitive way to represent relationships and dependencies among entities, making them a powerful tool for data organization and analysis.
___

## Example: Syntax Trees
Let's do an example where we parse a simple mathematical expression and visualize its concrete syntax tree (CST).

> __Concrete Syntax Tree (CST)__ A CST is a tree representation of the syntactic structure of source code. Each node of the tree represents a construct occurring in the source code. For example, we can have values, operators, and control flow constructs as nodes in an expression tree. We'll take advantage of [the `CSTParser.jl` package](https://github.com/julia-vscode/CSTParser.jl) to parse expressions. This is one of the packages associated with the Julia's vscode extension.

First, we set up the computational environment by including the `Include.jl` file and loading any needed resources. The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [27]:
include("Include.jl"); # loads all necessary packages and functions

Next, let's write a simple expression as a String, save it in the `simple_expression::String` variable:

In [28]:
simple_expression = "x^2+x+1";

Then, we'll call [the `parse(...)` method](https://www.julia-vscode.org/CSTParser.jl/dev/syntax/#CSTParser.parse) from [the `CSTParser.jl` package](https://github.com/julia-vscode/CSTParser.jl) to parse the expression and generate a CST.  The output of the parser will be stored in the `cst_expr::CSTParser.EXPR` variable:

In [29]:
cst_expr = CSTParser.parse(simple_expression)

  1:7   call
  1:1    OP: +
  2:4    call
  2:2     OP: ^
  3:3     x
  4:4     INTEGER: 2
  5:5    x
  6:6    INTEGER: 1

We can see the fields of the `cst_expr` variable by calling [the `fieldnames(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.fieldnames) on its type (which we get using [the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof)).

In [30]:
cst_expr |> typeof |> fieldnames

(:head, :args, :trivia, :fullspan, :span, :val, :parent, :meta)

Let's dig into these fields to understand the structure of our CST better. The tree represents the mathematical expression as a hierarchical structure:

In [31]:
cst_expr.head

:call

We can see the children of the root node (which represents the entire expression `x^2+x+1`):

In [32]:
length(cst_expr.args) # number of child nodes

4

The expression `x^2+x+1` is parsed as a tree where each operator becomes a parent node and its operands become children. Let's examine the structure by looking at the first few levels:

In [38]:
# Show the structure: root operator and its operands
println("Root: ", cst_expr.head)
for (i, arg) in enumerate(cst_expr.args)
    println("Child $i: ", arg.head, " -> ", arg.val)
end


Root: call
Child 1: OPERATOR -> +
Child 2: call -> nothing
Child 3: IDENTIFIER -> x
Child 4: INTEGER -> 1


___

<div>
    <center>
      <img
        src="figs/Fig-ThreeTree-Example-Schematic.svg"
        alt="Three Tree Example"
        height="600"
        width="800"
      />
    </center>
  </div>

## Full n-ary Trees
Ok, so trees model hierarchical relationships, but how do we store them? A common way to store trees is through __adjacency lists__. 

> __What is an adjacency list representation?__ In this representation, we split the data and the connectivity information into two separate structures. We store the data for each node in a linear collection of some sort, e.g., an array or perhaps a dictionary where the keys are node identifiers and the values are the node data. The connectivity is stored in a second list or array that contains the indices of the children for each node. This allows us to efficiently traverse the tree and access the children of any node.

Let's dig into the dimensionality of a complete $n$-ary tree, and the indexing scheme of the nodes in a tree. Suppose we have an $n$-ary tree, where each node has $n$ children. Further, suppose we say the root node has index 0. Then, for any node at index $i$, the $k$-th child where $k=1,2,\ldots,n$ can be found at the index:
$$
\begin{align*}
c_{i,k} = n\;i + k
\end{align*}
$$
while the parent of the node at index $i$ can be found at:
$$\begin{align*}
p_{i} = \left\lfloor \frac{i-1}{n} \right\rfloor
\end{align*}
$$
where $\lfloor x \rfloor$ is the floor function, which rounds down to the nearest integer. This indexing scheme allows us to efficiently navigate the tree structure and access parent-child relationships. This tree will have $N$ nodes where:
$$
\boxed{
\begin{align*}
N &= \sum_{i=0}^{h} n^{i} = \frac{n^{h+1}-1}{n-1}\quad \text{for } n > 1
\end{align*}}
$$
where $h$ is the height of the tree. Alternatively, we can write the number of nodes in a complete $n$-ary tree as a function of the number of levels $l$ in the tree, where $h = L - 1$:
$$\boxed{
\begin{align*}N &= \frac{n^{L}-1}{n-1}\quad \text{for } n > 1
\end{align*}}$$
The expression gives us the total number of nodes in a complete $n$-ary tree, which can be useful for understanding the storage requirements of the tree structure.

### Concrete Example: Ternary Tree
Let's apply these formulas to a concrete example. Consider a complete ternary tree ($n=3$) with height $h=2$ (3 levels total). We can calculate exactly how many nodes this tree contains and verify our indexing scheme:

In [34]:
# Ternary tree (n=3) with height h=2
n = 3  # number of children per node
h = 2  # height of tree

# Calculate total number of nodes using our formula
N = (n^(h+1) - 1) ÷ (n - 1)
println("Total nodes in ternary tree with height $h: $N")

# Verify by counting level by level
level_0 = n^0  # root level
level_1 = n^1  # first level children  
level_2 = n^2  # second level children
total_manual = level_0 + level_1 + level_2
println("Manual count: $level_0 + $level_1 + $level_2 = $total_manual")

# Test our indexing formulas
println("\nIndexing examples:")
println("Root node 0 has children at indices: ", [3*0 + k for k in 1:3])
println("Node 1 has children at indices: ", [3*1 + k for k in 1:3])  
println("Node 5 has parent at index: ", floor(Int, (5-1)/3))

Total nodes in ternary tree with height 2: 13
Manual count: 1 + 3 + 9 = 13

Indexing examples:
Root node 0 has children at indices: [1, 2, 3]
Node 1 has children at indices: [4, 5, 6]
Node 5 has parent at index: 1
[1, 2, 3]
Node 1 has children at indices: [4, 5, 6]
Node 5 has parent at index: 1


This demonstrates how our mathematical formulas work in practice: a ternary tree with height 2 contains exactly 13 nodes (1 root + 3 first-level + 9 second-level), and our indexing scheme allows us to efficiently navigate between parents and children using simple arithmetic operations.

### Example: A binary decision tree
Imagine we are designing a phone-based customer support system where callers are guided through a series yes/no questions. We can represent this process using a __binary decision tree__, where each internal node represents a question and each leaf node represents a final decision or outcome.

> __Binary decision tree:__ In this tree structure, the root node is the initial question, and each child node represents the possible answers to that question. For example, if the root question is "Is it raining?", the left child might represent the answer "yes" and the right child the answer "no". This branching continues until we reach a leaf node, which gives us the final decision based on the answers to all previous questions.

Let's build and populate a binary decision tree for our phone system. We'll use [a `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build) to construct a tree with its connectivity data specified, then we'll call [the `populate!(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.populate!), to add data to the tree. However, before we can do that, we'll define a helper __callback function__ to populate data of the tree nodes.

> __Callback function:__ A callback function is a function that is passed as an argument to another function and is executed at a later time, often in response to an event or condition. In this case, the callback function will be used to populate the tree nodes with data based on their indices and parent-child relationships.

We call our callback function `configuration(...)`, which takes the current node's index, an index for its child, and parent node data. The function returns a tuple containing the node's index, the child index, the question answer (true/false), and node path. The node path is the list of answers from the root (answer for the root is set to `-1` by convention) to the current node. 

This function is called for each node in the tree, and it constructs the path as it goes. The path is built by appending the current answer to the parent's path. The path is used to track the decisions made along the way, which can be useful for debugging or understanding the decision-making process.

In [35]:
function configuration(i, k, parentdata)::NamedTuple

    # What do we pass in?
    # i - my node index
    # k - my child index
    # parentdata - the data from my parent node, if any

    #  check: do we have any parent data?
    if (parentdata === nothing)
        parentdata = (i = -1, k = -1, answer = -1, path = [-1]) # root
        return parentdata;
    end
    
    # ok, so if we get here, we have parent data (we are not at the root)
    myvalue = isodd(k) == true ? 1 : 0 # if we are odd, we are 1, else we are 0
    
    # build the path to me from the root
    parent_path = parentdata.path # what is the path to my parent?
    mypath = vcat(parent_path, myvalue); # add my value to the path

    # i is the level, and k is the index at that level
    return (i = i, k = k, answer = myvalue, path = mypath)
end;

Now that we've defined our callback function, let's use it to build and populate our binary decision tree. The following code creates a decision tree model with the specific parameters defined above, and then populates it with data using our `configuration` callback function:

* We use the `build(...)` function to construct a `MyFullGeneralAdjacencyTree` with `n = 2` (binary tree) and `h = 3` (height of 3). After the tree is built, it contains the connectivity information (adjacency) of the nodes, but no data yet.
* The `populate!(...)` method then calls our `configuration` function for each node to assign data based on the tree structure
* This creates a complete binary tree where each node contains information about its position, answer value (0 or 1), and the complete path from the root

The resulting `decision_tree_model` will contain both the tree structure (adjacency information) and the node data (questions, answers, and paths) needed for our phone support system.

In [36]:
decision_tree_model = build(MyFullGeneralAdjacencyTree, (
    n = 2,
    h = 3,
)) |> m-> populate!(m, configuration);

Looking at the structure of `decision_tree_model`, it has two main fields:
* **`connectivity`**: Contains the adjacency information defining parent-child relationships between nodes. This field stores the tree structure as keys in the `connectivity::Dict{Int64,Array{Int64,1}}` dictionary, allowing efficient traversal from any node to its children or parent.

* **`data`**: Stores the actual content for each node, populated by our `configuration` callback function. Each node contains its index, child index, answer value (0 or 1 for left/right branches), and the complete path of decisions from the root to that node. This data is stored in a `data::Dict{Int64,NamedTuple}` dictionary, where the keys are node indices and the values are named tuples containing the node's data.

In [37]:
decision_tree_model.data[1]

(i = 0, k = 1, answer = 1, path = [-1, 1])